# Fine-Tuning a Pretrained LLM on a Domain Dataset
This notebook performs domain-specific sentiment classification fine-tuning using a Hugging Face pretrained model and includes PCA visualization of word vectors.

## 1) Identified the Task to Be Performed
Task: Multi-class text classification for financial sentiment (negative, neutral, positive).

In [1]:
import random
import numpy as np
import torch

from huggingface_hub import list_repo_files, hf_hub_download
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import plotly.graph_objects as go

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Libraries imported successfully.")

c:\Users\Acer\Documents\GitHub\CCS-249_25-26_Activities\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
Libraries imported successfully.


In [2]:
TASK = "Financial sentiment text classification"
DOMAIN = "Finance"

print(f"Task: {TASK}")
print(f"Domain: {DOMAIN}")

Task: Financial sentiment text classification
Domain: Finance


## 2) Identified the Domain Used for Fine-Tuning
Dataset: takala/financial_phrasebank from Hugging Face (finance-domain sentiment dataset).

In [3]:
import zipfile

repo_id = "takala/financial_phrasebank"
files = list_repo_files(repo_id=repo_id, repo_type="dataset")
print("Available files:", files)

zip_file = "data/FinancialPhraseBank-v1.0.zip"
if zip_file not in files:
    raise ValueError("Expected FinancialPhraseBank zip file not found.")

local_zip = hf_hub_download(repo_id=repo_id, repo_type="dataset", filename=zip_file)

texts = []
labels_str = []

with zipfile.ZipFile(local_zip, "r") as zf:
    target_name = None
    for name in zf.namelist():
        if "sentences_allagree.txt" in name.lower():
            target_name = name
            break

    if target_name is None:
        raise ValueError("Sentences_AllAgree.txt not found in zip archive.")

    with zf.open(target_name) as f:
        for raw_line in f:
            line = raw_line.decode("latin-1").strip()
            if not line or "@" not in line:
                continue
            text, label = line.rsplit("@", 1)
            texts.append(text.strip().strip('"'))
            labels_str.append(label.strip().lower())

preferred_order = ["negative", "neutral", "positive"]
unique_labels = [lbl for lbl in preferred_order if lbl in set(labels_str)]
if not unique_labels:
    unique_labels = sorted(list(set(labels_str)))

label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
id2label = {i: lbl for lbl, i in label2id.items()}
labels = [label2id[lbl] for lbl in labels_str]

# Pure Python stratified split (80/20)
indices_by_label = {}
for idx, lab in enumerate(labels):
    indices_by_label.setdefault(lab, []).append(idx)

train_indices, test_indices = [], []
for lab, idxs in indices_by_label.items():
    random.shuffle(idxs)
    split = int(0.8 * len(idxs))
    train_indices.extend(idxs[:split])
    test_indices.extend(idxs[split:])

random.shuffle(train_indices)
random.shuffle(test_indices)

train_texts = [texts[i] for i in train_indices]
train_labels = [labels[i] for i in train_indices]
test_texts = [texts[i] for i in test_indices]
test_labels = [labels[i] for i in test_indices]

print(f"Loaded samples: {len(texts)}")
print("Label mapping:", label2id)
print(f"Train size: {len(train_texts)} | Test size: {len(test_texts)}")

Available files: ['.gitattributes', 'README.md', 'data/FinancialPhraseBank-v1.0.zip', 'financial_phrasebank.py']


Loaded samples: 2264
Label mapping: {'negative': 0, 'neutral': 1, 'positive': 2}
Train size: 1810 | Test size: 454


## 3) Identified the LLM to Be Used
Model: distilbert-base-uncased (pretrained transformer from Hugging Face for sequence classification fine-tuning).

In [4]:
MODEL_CHECKPOINT = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print("Loaded tokenizer for:", MODEL_CHECKPOINT)
print("Classes:", id2label)

Loaded tokenizer for: distilbert-base-uncased
Classes: {0: 'negative', 1: 'neutral', 2: 'positive'}


## 4) Established the Configuration Needed for Fine-Tuning
This section prepares train/test splits, tokenization, Trainer arguments, and launches fine-tuning.

In [5]:
# Subsample for faster local training if needed
max_train = min(1200, len(train_texts))
max_test = min(300, len(test_texts))

train_texts_small = train_texts[:max_train]
train_labels_small = train_labels[:max_train]
test_texts_small = test_texts[:max_test]
test_labels_small = test_labels[:max_test]

class EncodedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=max_length)
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = EncodedTextDataset(train_texts_small, train_labels_small, tokenizer)
test_dataset = EncodedTextDataset(test_texts_small, test_labels_small, tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
).to(device)

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7618.53it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train size: 1200
Test size: 300


In [6]:
def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

def macro_f1_np(y_true, y_pred, num_classes):
    cm = confusion_matrix_np(y_true, y_pred, num_classes)
    f1_scores = []
    for i in range(num_classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)
    return float(np.mean(f1_scores))

# Fine-tuning configuration
num_epochs = 2
learning_rate = 2e-5
train_batch_size = 16
eval_batch_size = 16
weight_decay = 0.01

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=eval_batch_size, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

print("Training config:")
print({
    "num_epochs": num_epochs,
    "learning_rate": learning_rate,
    "train_batch_size": train_batch_size,
    "eval_batch_size": eval_batch_size,
    "weight_decay": weight_decay,
})

Training config:
{'num_epochs': 2, 'learning_rate': 2e-05, 'train_batch_size': 16, 'eval_batch_size': 16, 'weight_decay': 0.01}


In [7]:
best_f1 = -1.0
best_state = None

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / max(1, len(train_loader))

    # Evaluate each epoch
    model.eval()
    y_true_epoch, y_pred_epoch = [], []

    with torch.no_grad():
        for batch in test_loader:
            labels = batch["labels"].numpy()
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            preds = torch.argmax(logits, dim=1).cpu().numpy()

            y_true_epoch.extend(labels.tolist())
            y_pred_epoch.extend(preds.tolist())

    epoch_acc = float((np.array(y_true_epoch) == np.array(y_pred_epoch)).mean())
    epoch_f1 = macro_f1_np(np.array(y_true_epoch), np.array(y_pred_epoch), len(id2label))

    print(f"Epoch {epoch}/{num_epochs} | train_loss={avg_train_loss:.4f} | val_acc={epoch_acc:.4f} | val_f1_macro={epoch_f1:.4f}")

    if epoch_f1 > best_f1:
        best_f1 = epoch_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)

print("Training finished.")
print("Best validation macro-F1:", round(best_f1, 4))

Epoch 1/2 | train_loss=0.6581 | val_acc=0.9100 | val_f1_macro=0.8760
Epoch 2/2 | train_loss=0.2044 | val_acc=0.9433 | val_f1_macro=0.9177
Training finished.
Best validation macro-F1: 0.9177


## 5) Performed Evaluation
This section reports accuracy/F1, classification report, and confusion matrix on the held-out test split (negative vs positive).

In [8]:
model.eval()
y_true_list, y_pred_list = [], []

with torch.no_grad():
    for batch in test_loader:
        labels = batch["labels"].numpy()
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()

        y_true_list.extend(labels.tolist())
        y_pred_list.extend(preds.tolist())

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)

num_classes = len(id2label)
cm = np.zeros((num_classes, num_classes), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

acc = float((y_true == y_pred).mean())

# Macro-F1 (NumPy implementation)
f1_per_class = []
for i in range(num_classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    f1_per_class.append(f1)

f1m = float(np.mean(f1_per_class))

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {f1m:.4f}")
print("\nPer-class metrics:")
for i in range(num_classes):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    support = cm[i, :].sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
    print(f"{id2label[i]:>8} | precision={precision:.4f} recall={recall:.4f} f1={f1:.4f} support={support}")

labels_axis = [id2label[i] for i in range(num_classes)]
fig_cm = go.Figure(data=go.Heatmap(
    z=cm,
    x=labels_axis,
    y=labels_axis,
    colorscale="Blues",
    showscale=True,
    text=cm,
    texttemplate="%{text}"
))
fig_cm.update_layout(
    title="Confusion Matrix - Financial Sentiment",
    xaxis_title="Predicted",
    yaxis_title="Actual"
)
fig_cm.show()

Accuracy: 0.9433
Macro F1: 0.9177

Per-class metrics:
negative | precision=0.9091 recall=0.8824 f1=0.8955 support=34
 neutral | precision=0.9502 recall=0.9948 f1=0.9720 support=192
positive | precision=0.9394 recall=0.8378 f1=0.8857 support=74


## 6) PCA: Visualize Vectors for at Least 20 Known Words
This section extracts token embeddings from the fine-tuned model and projects them to 2D using PCA.

In [9]:
known_words = [
    "market", "stock", "profit", "loss", "growth", "revenue", "bank", "loan", "credit", "debt",
    "trade", "inflation", "economy", "investment", "risk", "dividend", "share", "fund", "price", "demand",
    "supply", "cost", "asset", "liability"
]

embedding_layer = model.get_input_embeddings().weight.detach().cpu().numpy()

word_vectors = []
valid_words = []

for word in known_words:
    tokens = tokenizer.tokenize(word)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    token_ids = [tid for tid in token_ids if tid != tokenizer.unk_token_id]

    if len(token_ids) == 0:
        continue

    # Average subword vectors to represent the whole word.
    vec = embedding_layer[token_ids].mean(axis=0)
    word_vectors.append(vec)
    valid_words.append(word)

X = np.array(word_vectors)
X_centered = X - X.mean(axis=0, keepdims=True)

# PCA via eigen decomposition of covariance matrix (no sklearn dependency).
cov = np.cov(X_centered, rowvar=False)
eigvals, eigvecs = np.linalg.eigh(cov)
principal_axes = eigvecs[:, -2:]
vec_2d = X_centered @ principal_axes

fig_pca = go.Figure(data=go.Scatter(
    x=vec_2d[:, 0],
    y=vec_2d[:, 1],
    mode="markers+text",
    text=valid_words,
    textposition="top center",
    marker=dict(size=10, color=list(range(len(valid_words))), colorscale="Viridis", showscale=True)
))
fig_pca.update_layout(
    title=f"PCA of Fine-Tuned Embeddings ({len(valid_words)} known words)",
    xaxis_title="Principal Component 1",
    yaxis_title="Principal Component 2"
)
fig_pca.show()

print("Words used for PCA:", len(valid_words))
print(valid_words)

Words used for PCA: 24
['market', 'stock', 'profit', 'loss', 'growth', 'revenue', 'bank', 'loan', 'credit', 'debt', 'trade', 'inflation', 'economy', 'investment', 'risk', 'dividend', 'share', 'fund', 'price', 'demand', 'supply', 'cost', 'asset', 'liability']


## 7) Submission Notes
Take screenshots of: dataset preview, training logs, evaluation metrics, confusion matrix, and PCA plot.